# Mini-PLSC Tutorial

**Partial Least Squares Correlation (PLSC)** finds pairs of linear projections — one from matrix X, one from matrix Y — that maximally correlate.  Unlike CCA it does not require X and Y to be square or well-conditioned.

Here X and Y are the RNN states of agent A and agent B at matched timesteps.  PLSC asks: _how much shared variance do the two agents' internal representations carry?_

The tutorial below uses synthetic data so you can verify the method recovers a known ground truth before running it on real RNN states.

In [ ]:
import sys, os, time
sys.path.insert(0, '/home/satsingh/kr/mfrefactor/onpolicy/custom/fish')
os.chdir('/home/satsingh/kr/mfrefactor/onpolicy/custom/fish')
import importlib
import analysis_rnn_plsc as m; importlib.reload(m)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.cross_decomposition import PLSCanonical
from scipy.stats import pearsonr

### Synthetic data with *k* shared latent dimensions

In [ ]:
rng = np.random.default_rng(42)
n, p, q, k = 300, 20, 20, 3   # n timesteps; p/q hidden-unit dims; k shared dims
noise = 0.5

# k shared latent factors drive both X and Y
Z  = rng.standard_normal((n, k))
Wx = rng.standard_normal((k, p)); Wx /= np.linalg.norm(Wx, axis=1, keepdims=True)
Wy = rng.standard_normal((k, q)); Wy /= np.linalg.norm(Wy, axis=1, keepdims=True)
X  = Z @ Wx + noise * rng.standard_normal((n, p))   # agent A states
Y  = Z @ Wy + noise * rng.standard_normal((n, q))   # agent B states
print(f"X: {X.shape}  Y: {Y.shape}  ({k} shared latent dims, noise={noise})")

### Run `compute_plsc` and check recovered dimension count

In [ ]:
result = m.compute_plsc(X, Y, n_components=8, n_shuffles=50, seed=42)
print(f"num_sig  : {result['num_sig']}   (true k = {k})")
print(f"top_corr : {result['top_corr']:.3f}")
print(f"all corrs: {[f'{r:.3f}' for r in result['correlations']]}")

### Speed check at realistic RNN dimensions (H=128)

In [ ]:
configs = [
    # label,                                     n,    H,   nc, ns
    ('n=40,  H=128, nc=1,  ns=0   (distance loop, no shuffle)', 40,   128,  1,  0),
    ('n=40,  H=128, nc=10, ns=30  (sig-dims, small ep)',        40,   128, 10, 30),
    ('n=300, H=128, nc=10, ns=30  (sig-dims, typical ep)',     300,   128, 10, 30),
    ('n=1000,H=128, nc=10, ns=30  (sig-dims, max_samples)',   1000,   128, 10, 30),
]
rng2 = np.random.default_rng(0)
for label, n, H, nc, ns in configs:
    Xb = rng2.standard_normal((n, H))
    Yb = rng2.standard_normal((n, H))
    t0 = time.perf_counter()
    m.compute_plsc(Xb, Yb, n_components=nc, n_shuffles=ns, seed=42)
    print(f"{time.perf_counter()-t0:6.2f}s  {label}")

### Canonical correlations vs null distribution

In [ ]:
corrs    = np.array(result['correlations'])
null_95  = np.percentile(result['null_corrs'], 97.5, axis=0)
n_comps  = len(corrs)

fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(range(n_comps), corrs, color='steelblue', alpha=0.8, label='PLSC canonical corr')
ax.step(range(n_comps), null_95, where='mid', color='tomato', lw=1.5,
        linestyle='--', label='97.5th pct null')
ax.set_xlabel('Component index')
ax.set_ylabel('Canonical correlation')
ax.set_title(f'{result["num_sig"]} significant dims  (true k={k})')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

### First canonical variates — agent A vs agent B projections

In [ ]:
Xc, Yc = X - X.mean(0), Y - Y.mean(0)
plsc_fit = PLSCanonical(n_components=k).fit(Xc, Yc)
Xs, Ys  = plsc_fit.transform(Xc, Yc)

fig, axes = plt.subplots(1, k, figsize=(3.5 * k, 3.2))
for i, ax in enumerate(axes):
    r, _ = pearsonr(Xs[:, i], Ys[:, i])
    ax.scatter(Xs[:, i], Ys[:, i], s=10, alpha=0.45, color='steelblue')
    ax.set_title(f'Component {i+1}  r={r:.2f}')
    ax.set_xlabel('Agent A canonical variate')
    if i == 0:
        ax.set_ylabel('Agent B canonical variate')
plt.suptitle('Canonical variates (paired RNN projections)', y=1.02)
plt.tight_layout()
plt.show()

### Null check — independent X and Y should give 0 significant dims

In [ ]:
X_rand  = rng.standard_normal((n, p))
Y_rand  = rng.standard_normal((n, q))
res_rnd = m.compute_plsc(X_rand, Y_rand, n_components=8, n_shuffles=50, seed=42)
print(f"Random data — num_sig: {res_rnd['num_sig']}  top_corr: {res_rnd['top_corr']:.3f}")

---
# Analysis — RNN PLSC on real eval data

In [ ]:
# ── CONFIG — edit these lines ─────────────────────────────────────────────────
RUN_DIR      = '~/cluster_lab/satsingh/marl_fish_storage/results/ConsNoise20260608dynamicT5MFO0.1FX1.0Order1LinearX2.0AngularX4.0Gamma0.995DCL100TD0.0PR5UR1NP0A1K1M1GRU/seed1/20XXXXXX_XXXXXX'
SPEC_KEY     = '2fish_m1a1k1_uniform_wide'
MORM_RANGE_CM = 10.0

In [ ]:
spec_dir = os.path.join(os.path.expanduser(RUN_DIR), 'evals', SPEC_KEY)
data = m.load(spec_dir, morm_range_cm=MORM_RANGE_CM)

## Fig 6G — Number of significant shared PLSC dimensions (in-range vs out-of-range)

In [ ]:
m.plot_sig_dims(data['sig_df'], data['base'])

## PLSC1 canonical correlation (in-range vs out-of-range)

In [ ]:
m.plot_plsc1_by_range(data['sig_df'], data['base'])